In [9]:
import jax
import jax.experimental.pallas as pl

import numpy as np
import jax.numpy as jnp

def show_program_ids(x_shape, block_shape, grid,
                    index_map=lambda i, j: (i, j),
                    indexing_mode=pl.Blocked()):
    def program_ids_kernel(o_ref):  # Fill the output block with 10*program_id(1) + program_id(0)
        axes = 0
        for axis in range(len(grid)):
            axes += pl.program_id(axis) * 10**(len(grid) - 1 - axis)
        o_ref[...] = jnp.full(o_ref.shape, axes)
    res = pl.pallas_call(program_ids_kernel,
        out_shape=jax.ShapeDtypeStruct(x_shape, dtype=np.int32),
        grid=grid,
        in_specs=[],
        out_specs=pl.BlockSpec(block_shape, index_map, indexing_mode=indexing_mode),
        interpret=True)()
    print(res)


In [ ]:
def show_program_ids(x_shape, block_shape, grid,
                    index_map=lambda i, j: (i, j),
                    indexing_mode=pl.Blocked()):
    def program_ids_kernel(o_ref):  # Fill the output block with 10*program_id(1) + program_id(0)
        axes = 0
        for axis in range(len(grid)):
            axes += pl.program_id(axis) * 10**(len(grid) - 1 - axis)
        o_ref[...] = jnp.full(o_ref.shape, axes)
    res = pl.pallas_call(program_ids_kernel,
        out_shape=jax.ShapeDtypeStruct(x_shape, dtype=np.int32),
        grid=grid,
        in_specs=[],
        out_specs=pl.BlockSpec(block_shape, index_map, indexing_mode=indexing_mode),
        interpret=True)()
    print(res)

In [11]:
show_program_ids((8, 8), (2, 2), (4, 4))

[[ 0  0  1  1  2  2  3  3]
 [ 0  0  1  1  2  2  3  3]
 [10 10 11 11 12 12 13 13]
 [10 10 11 11 12 12 13 13]
 [20 20 21 21 22 22 23 23]
 [20 20 21 21 22 22 23 23]
 [30 30 31 31 32 32 33 33]
 [30 30 31 31 32 32 33 33]]


In [15]:
show_program_ids((8,), (2,), (4,), index_map=lambda i,: (i,))

[0 0 1 1 2 2 3 3]


In [37]:
import jax
import jax.numpy as jnp
import numpy as np
import jax.experimental.pallas as pl

def compute_average_distance_between_X_and_Y_pallas(X, Y,
                                                    block_size_b=2,
                                                    block_size_d=2):
    """
    Single Pallas kernel to compute average pairwise L1 distances between
    rows of X and rows of Y, each of shape (B, D).

    Steps:
      1) Build a (B,B) 'dist_matrix' with dist_matrix[i,j] = sum_k |X[i,k] - Y[j,k]|.
      2) Divide by D to average across features.
      3) Return dist_matrix.mean(axis=0), shape (B,).

    If you want a single scalar that is the average distance across *all* pairs,
    you could further do dist_matrix.mean() or similar.
    """
    B, D = X.shape
    # Basic checks for shapes
    assert Y.shape == (B, D), "Y must match X's shape (B, D)."

    # We'll accumulate partial sums into a (B,B) output. Then do final scaling by D and .mean(0).
    dist_init = jnp.zeros((B, B), dtype=X.dtype)

    def single_kernel(x_ref, y_ref, out_ref):
        """
        We launch a 3D grid:
          - gridDimX => block-tiling of the 'i' index in [0..B).
          - gridDimY => block-tiling of the 'j' index in [0..B).
          - gridDimZ => block-tiling of the 'd' index in [0..D).

        Each tile accumulates partial sums of |X[i_idx, d_idx] - Y[j_idx, d_idx]|.
        """
        tile_i = pl.program_id(0)  # which block along i dimension
        tile_j = pl.program_id(1)  # which block along j dimension
        tile_d = pl.program_id(2)  # which block along d dimension

        i_start = tile_i * block_size_b
        j_start = tile_j * block_size_b
        d_start = tile_d * block_size_d

        for bi in range(block_size_b):
            for bj in range(block_size_b):
                for bd in range(block_size_d):
                    i_idx = i_start + bi
                    j_idx = j_start + bj
                    d_idx = d_start + bd

                    # Check bounds for all three indices.
                    valid = (i_idx < B) & (j_idx < B) & (d_idx < D)

                    # Safely load from X[i_idx, d_idx] and Y[j_idx, d_idx]
                    x_val = jnp.where(
                        valid,
                        pl.load(x_ref, (bi, bd)),  # local tile coords
                        0.0
                    )
                    y_val = jnp.where(
                        valid,
                        pl.load(y_ref, (bj, bd)),  # local tile coords
                        0.0
                    )

                    diff = jnp.abs(x_val - y_val)

                    # Atomic-add partial difference into out[i_idx, j_idx]
                    # if within valid range.
                    if valid:
                        pl.atomic_add(out_ref, (i_idx, j_idx), diff)

    # 3D grid tiling:
    #   dimension 0 => B in steps of block_size_b
    #   dimension 1 => B in steps of block_size_b
    #   dimension 2 => D in steps of block_size_d
    grid = (
        (B + block_size_b - 1) // block_size_b,
        (B + block_size_b - 1) // block_size_b,
        (D + block_size_d - 1) // block_size_d
    )

    # ---------- Block Specs ----------

    # 1) X-spec: local tile shape is (block_size_b, block_size_d).
    #    The tile (tile_i, tile_j, tile_d) doesn't "use" tile_j for X;
    #    we only need 'tile_i' and 'tile_d' in the index_map for X.
    x_block_spec = pl.BlockSpec(
        index_map=lambda bi, bd: (
            pl.program_id(0) * block_size_b + bi,  # i in [0..B)
            pl.program_id(2) * block_size_d + bd   # d in [0..D)
        ),
        block_shape=(block_size_b, block_size_d)
    )

    # 2) Y-spec: local tile shape is (block_size_b, block_size_d).
    #    We only need tile_j and tile_d for Y's row/feature index.
    y_block_spec = pl.BlockSpec(
        index_map=lambda bj, bd: (
            pl.program_id(1) * block_size_b + bj,  # j in [0..B)
            pl.program_id(2) * block_size_d + bd   # d in [0..D)
        ),
        block_shape=(block_size_b, block_size_d)
    )

    # 3) Output spec for (B,B). The tile uses tile_i, tile_j in the index_map,
    #    ignoring tile_d, because partial sums over D happen via atomic_add.
    out_block_spec = pl.BlockSpec(
        index_map=lambda bi, bj: (
            pl.program_id(0) * block_size_b + bi,  # i in [0..B)
            pl.program_id(1) * block_size_b + bj   # j in [0..B)
        ),
        block_shape=(block_size_b, block_size_b)
    )

    out_shape = jax.ShapeDtypeStruct((B, B), X.dtype)

    # Launch the single kernel with two input specs (X and Y),
    # plus one output spec (dist_matrix).
    dist_matrix = pl.pallas_call(
        single_kernel,
        out_shape=out_shape,
        grid=grid,
        in_specs=[x_block_spec, y_block_spec],
        out_specs=[out_block_spec],
        interpret=True
    )(X, Y, dist_init)

    # dist_matrix[i,j] now holds sum_{k=0..D-1} |X[i,k] - Y[j,k]|.
    # Convert to "average distance across features" => / D,
    # then .mean(0) => shape (B,).
    result = (dist_matrix / D).mean(axis=0)
    return result


# ------------------------------------------------------------------------------
# Example usage & correctness check:
if __name__ == "__main__":
    key = jax.random.PRNGKey(0)
    B, D = 5, 3
    X = jax.random.normal(key, shape=(B, D))
    Y = jax.random.normal(key, shape=(B, D))

    # Single-kernel pallas result
    pallas_res = compute_average_distance_between_X_and_Y_pallas(
        X, Y, block_size_b=2, block_size_d=2
    )

    # Reference computation:
    # For each (i,j), sum_k |X[i,k] - Y[j,k]| / D
    # Then take .mean(axis=0) -> shape (B,).
    ref_matrix = jnp.abs(X[:, None, :] - Y[None, :, :]).sum(axis=-1) / D
    ref_res = ref_matrix.mean(axis=0)

    print("Pallas result:", pallas_res)
    print("Reference   :", ref_res)
    print("All close?  :", jnp.allclose(pallas_res, ref_res))

ValueError: Pytree for `in_specs` and inputs do not match. There are 1 mismatches, including:
    * `in_specs` is a tuple of length 2 but inputs is a tuple of length 3, so the lengths do not match

In [33]:
X = jax.random.normal(jax.random.PRNGKey(0), (10, 1))

In [34]:
compute_average_diff(X)

Array([1.4978703 , 1.8199685 , 0.8622444 , 0.7912489 , 0.7912489 ,
       1.1773854 , 0.886926  , 0.8549065 , 0.92289466, 1.1598449 ],      dtype=float32)

In [36]:
compute_average_diff_pallas_single_kernel(X)

ValueError: Pytree for `in_specs` and inputs do not match. There are 1 mismatches, including:
    * `in_specs` is a tuple of length 1 but inputs is a tuple of length 2, so the lengths do not match